In [1]:
from google.colab import drive
from datasets import load_from_disk
import pandas as pd

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Replace the string below with the path you just copied!
folder_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/combined_datasets_lite_hf"

# Load the dataset
combined_dataset = load_from_disk(folder_path)

# Convert to pandas
df_combined = combined_dataset.to_pandas()
print(df_combined.head())

In [4]:
csv_a_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_lite_Trek.csv"
csv_b_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Brandon.csv"

df_a = pd.read_csv(csv_a_path)
df_b = pd.read_csv(csv_b_path)

# --------- Normalize text (important!) ---------
def normalize(df):
    for col in ["question", "answer", "type"]:
        df[col] = df[col].astype(str).str.strip().str.lower()
    return df

df_a = normalize(df_a)
df_b = normalize(df_b)

# --------- Create duplicate keys ---------
df_a["key"] = list(zip(df_a["question"], df_a["answer"], df_a["type"]))
df_b["key"] = list(zip(df_b["question"], df_b["answer"], df_b["type"]))

# --------- Find duplicates ---------
duplicates_mask = df_a["key"].isin(df_b["key"])

duplicates = df_a[duplicates_mask]
filtered_df_a = df_a[~duplicates_mask]

# --------- Report ---------
print(f"Total rows in CSV A: {len(df_a)}")
print(f"Total rows in CSV B: {len(df_b)}")
print(f"Duplicates found in A (present in B): {len(duplicates)}")
print(f"Remaining rows after removal: {len(filtered_df_a)}")

# Optional: show some duplicates
print("\nSample duplicates:")
print(duplicates.head())

# --------- Save outputs ---------
filtered_df_a.drop(columns=["key"]).to_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_lite_deduplicater.csv", index=False)

print("\nSaved:")
print("- csv_a_deduplicated.csv (A without duplicates)")
print("- duplicates_found.csv (overlap between A and B)")

Total rows in CSV A: 498
Total rows in CSV B: 498
Duplicates found in A (present in B): 48
Remaining rows after removal: 450

Sample duplicates:
                                             question         answer  \
14  vicky yang and tamika harris both spoke to the...            nan   
31      who wrote the novel gentlemen prefer blondes?     anita loos   
39  who beat jim brown's rushing yards total of 12...  walter payton   
75         who sang a crazy little thing called love?          queen   
81  what was clive sinclair's personal transport v...             c5   

                             question_id     type  \
14  fc533799-ef37-460c-82bb-24c6b584de67  abstain   
31  7f8562a6-67e3-42b6-a908-5f44e5f627fb   answer   
39  c7d9417b-5ef7-4cbb-a8ac-96f57619c3ae   answer   
75  55f76510-56f0-427a-9166-f1e415342e31  clarify   
81  4067d49a-054e-4be5-816a-d4f573bfe92c   answer   

                                                  key  
14  (vicky yang and tamika harris both spoke to

In [5]:
import pandas as pd

# --------- File paths ---------
csv_c_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_lite_deduplicater.csv"
csv_d_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_lite_Brandon.csv"
csv_b_path = "/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Brandon.csv"

target_total_rows = 502  # <-- set your desired Y

# --------- Load ---------
df_c = pd.read_csv(csv_c_path)
df_d = pd.read_csv(csv_d_path)
df_b = pd.read_csv(csv_b_path)

# --------- Normalize ---------
def normalize(df):
    for col in ["question", "answer", "type"]:
        df[col] = df[col].fillna("").astype(str).str.strip().str.lower()
    return df

df_c = normalize(df_c)
df_d = normalize(df_d)
df_b = normalize(df_b)

# --------- Create keys ---------
def make_keys(df):
    return list(zip(df["question"], df["answer"], df["type"]))

c_keys = set(make_keys(df_c))
b_keys = set(make_keys(df_b))
d_keys = make_keys(df_d)

# --------- Filter D ---------
# Keep rows in D that are NOT in B and NOT already in C
valid_rows = df_d[
    ~pd.Series(d_keys).isin(b_keys | c_keys)
].copy()

# --------- Compute how many to add ---------
current_size = len(df_c)
rows_needed = target_total_rows - current_size

if rows_needed <= 0:
    print("CSV C already has >= target rows. No rows added.")
    final_df = df_c
else:
    print(f"Need to add {rows_needed} rows.")

    if len(valid_rows) < rows_needed:
        print(f"Warning: Only {len(valid_rows)} valid rows available in D.")
        rows_to_add = valid_rows
    else:
        # Sample randomly (remove random_state for true randomness)
        rows_to_add = valid_rows.sample(n=rows_needed, random_state=42)

    # --------- Append ---------
    final_df = pd.concat([df_c, rows_to_add], ignore_index=True)

# --------- Save ---------
final_df.to_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/csv_c_augmented.csv", index=False)

print(f"Final dataset size: {len(final_df)}")
print("Saved to csv_c_augmented.csv")

Need to add 52 rows.
Final dataset size: 502
Saved to csv_c_augmented.csv


In [8]:
import pandas as pd

# --------- Load ---------
df_a = pd.read_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Trek.csv")
df_b = pd.read_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Brandon.csv")

# --------- Normalize ---------
def normalize(df):
    for col in ["question", "type"]:
        df[col] = df[col].fillna("").astype(str).str.strip().str.lower()
    return df

df_a = normalize(df_a)
df_b = normalize(df_b)

# --------- Duplicate Checks (QUESTION LEVEL) ---------
dup_q_a = df_a["question"].duplicated().sum()
dup_q_b = df_b["question"].duplicated().sum()

print("=== DUPLICATE QUESTIONS (within each CSV) ===")
print(f"CSV A duplicate questions: {dup_q_a}")
print(f"CSV B duplicate questions: {dup_q_b}")

# --------- Deduplicated versions (based on question) ---------
df_a_unique = df_a.drop_duplicates(subset=["question"])
df_b_unique = df_b.drop_duplicates(subset=["question"])

# --------- Basic Counts ---------
print("\n=== ROW COUNTS ===")
print(f"CSV A (raw): {len(df_a)}")
print(f"CSV B (raw): {len(df_b)}")
print(f"Total (raw): {len(df_a) + len(df_b)}")

print(f"\nCSV A (unique questions): {len(df_a_unique)}")
print(f"CSV B (unique questions): {len(df_b_unique)}")
print(f"Total (unique): {len(df_a_unique) + len(df_b_unique)}")

# --------- Type Breakdown (Individual, RAW) ---------
print("\n=== TYPE BREAKDOWN (RAW) ===")
print("\nCSV A:")
print(df_a["type"].value_counts())

print("\nCSV B:")
print(df_b["type"].value_counts())

# --------- Type Breakdown (Individual, UNIQUE) ---------
print("\n=== TYPE BREAKDOWN (UNIQUE QUESTIONS ONLY) ===")
print("\nCSV A:")
print(df_a_unique["type"].value_counts())

print("\nCSV B:")
print(df_b_unique["type"].value_counts())

# --------- Combined (RAW) ---------
df_combined = pd.concat([df_a, df_b], ignore_index=True)
print("\n=== TYPE BREAKDOWN (COMBINED RAW) ===")
print(df_combined["type"].value_counts())

# --------- Combined (UNIQUE QUESTIONS) ---------
df_combined_unique = pd.concat([df_a_unique, df_b_unique], ignore_index=True)
df_combined_unique = df_combined_unique.drop_duplicates(subset=["question"])

print("\n=== TYPE BREAKDOWN (COMBINED UNIQUE QUESTIONS) ===")
print(df_combined_unique["type"].value_counts())

# --------- Side-by-side comparison (UNIQUE) ---------
comparison_unique = pd.DataFrame({
    "A_unique": df_a_unique["type"].value_counts(),
    "B_unique": df_b_unique["type"].value_counts(),
    "Combined_unique": df_combined_unique["type"].value_counts()
}).fillna(0).astype(int)

print("\n=== SIDE-BY-SIDE (UNIQUE QUESTIONS) ===")
print(comparison_unique)

=== DUPLICATE QUESTIONS (within each CSV) ===
CSV A duplicate questions: 0
CSV B duplicate questions: 0

=== ROW COUNTS ===
CSV A (raw): 502
CSV B (raw): 498
Total (raw): 1000

CSV A (unique questions): 502
CSV B (unique questions): 498
Total (unique): 1000

=== TYPE BREAKDOWN (RAW) ===

CSV A:
type
abstain    168
answer     167
clarify    167
Name: count, dtype: int64

CSV B:
type
clarify    166
abstain    166
answer     166
Name: count, dtype: int64

=== TYPE BREAKDOWN (UNIQUE QUESTIONS ONLY) ===

CSV A:
type
abstain    168
answer     167
clarify    167
Name: count, dtype: int64

CSV B:
type
clarify    166
abstain    166
answer     166
Name: count, dtype: int64

=== TYPE BREAKDOWN (COMBINED RAW) ===
type
abstain    334
answer     333
clarify    333
Name: count, dtype: int64

=== TYPE BREAKDOWN (COMBINED UNIQUE QUESTIONS) ===
type
abstain    334
answer     333
clarify    333
Name: count, dtype: int64

=== SIDE-BY-SIDE (UNIQUE QUESTIONS) ===
         A_unique  B_unique  Combined_unique

In [7]:
import pandas as pd

# --------- Load ---------
df_a = pd.read_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Trek.csv")
df_b = pd.read_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_Brandon.csv")
df_c = pd.read_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/misc/dataset_lite_Brandon.csv")

# --------- Normalize ---------
def normalize(df):
    for col in ["question", "answer", "type"]:
        df[col] = df[col].fillna("").astype(str).str.strip().str.lower()
    return df

df_a = normalize(df_a)
df_b = normalize(df_b)
df_c = normalize(df_c)

# --------- Key function ---------
def make_key(df):
    return list(zip(df["question"], df["answer"], df["type"]))

a_keys = set(make_key(df_a))
b_keys = set(make_key(df_b))

# --------- Type info ---------
types = sorted(df_a["type"].unique())
n_types = len(types)
total_size = len(df_a)

base_target = total_size // n_types
remainder = total_size % n_types

# Assign targets (some get +1)
target_counts = {}
for i, t in enumerate(types):
    target_counts[t] = base_target + (1 if i < remainder else 0)

print("Target per type:", target_counts)

# --------- Split A by type ---------
grouped_a = {t: df_a[df_a["type"] == t].copy() for t in types}

kept_rows = []
needed_counts = {}

# --------- Step 1: Downsample large classes ---------
for t in types:
    current = len(grouped_a[t])
    target = target_counts[t]

    if current > target:
        # remove excess
        kept = grouped_a[t].sample(n=target, random_state=42)
        kept_rows.append(kept)
    else:
        kept_rows.append(grouped_a[t])
        needed_counts[t] = target - current  # how many we need to add

# Update A keys after removal
df_a_reduced = pd.concat(kept_rows, ignore_index=True)
a_reduced_keys = set(make_key(df_a_reduced))

# --------- Step 2: Filter C (no duplicates with A or B) ---------
c_keys = make_key(df_c)
valid_c_mask = ~pd.Series(c_keys).isin(a_reduced_keys | b_keys)
df_c_valid = df_c[valid_c_mask].copy()

# --------- Step 3: Add rows for underrepresented types ---------
added_rows = []

for t, needed in needed_counts.items():
    if needed == 0:
        continue

    candidates = df_c_valid[df_c_valid["type"] == t]

    if len(candidates) < needed:
        print(f"Warning: Not enough samples in C for type '{t}'. Needed {needed}, found {len(candidates)}")
        selected = candidates
    else:
        selected = candidates.sample(n=needed, random_state=42)

    added_rows.append(selected)

# --------- Final dataset ---------
df_final = pd.concat([df_a_reduced] + added_rows, ignore_index=True)

# --------- Final check ---------
print("\nFinal size:", len(df_final))
print("\nFinal distribution:")
print(df_final["type"].value_counts())

# --------- Save ---------
df_final.to_csv("/content/drive/MyDrive/CSCI 5980 8980 Project/Notebooks/Data/dataset_lite_Trek.csv", index=False)
print("\nSaved to csv_a_balanced.csv")

Target per type: {'abstain': 168, 'answer': 167, 'clarify': 167}

Final size: 502

Final distribution:
type
abstain    168
answer     167
clarify    167
Name: count, dtype: int64

Saved to csv_a_balanced.csv
